In [1]:
!pip install adversarial-robustness-toolbox -q
"""
================================================================================
APOLLON Baseline - CIC-IDS2017 Adversarial Evaluation 
================================================================================
Scenario: Replicate APOLLON (zoo.ipynb) with attack settings from scenario2.md
================================================================================

This script implements APOLLON defense system exactly as in zoo.ipynb:
- Models: RandomForest, MLP, GaussianNB, DecisionTree, LogisticRegression
- MAB: Thompson Sampling with KMeans clustering
- Attacks: ZOO + HopSkipJump (same as scenario2.md)

APOLLON Reference:
    "APOLLON: A Robust Defence System against Adversarial Machine Learning 
     Attacks in Intrusion Detection Systems"
    - Uses Multi-Armed Bandits (MAB) with Thompson Sampling
    - KMeans clustering to segment input space
    - Dynamically selects optimal classifier per sample/cluster

================================================================================
"""

import os
import sys
import json
import time
import copy
import logging
import warnings
from typing import Dict, List, Tuple, Optional
from collections import deque

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier as skMLP
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, recall_score, precision_score, 
                             confusion_matrix, roc_auc_score, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# =============================================================================
# TPU/GPU DETECTION
# =============================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🎯 Using Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

# =============================================================================
# REPRODUCIBILITY - SEED=42
# =============================================================================
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# =============================================================================
# CIC-IDS2017 PATHS (Same as scenario2.md)
# =============================================================================

DATA_DIR = "/kaggle/input/cic-ids2017/MachineLearningCVE"
CSV_FILES = [
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
]

OUTPUT_DIR = "/kaggle/working"
MODEL_DIR = "/kaggle/working/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

DROP_COLS = [
    "Flow ID",    
    'Fwd Header Length.1',
    "Source IP", "Src IP",
    "Source Port", "Src Port",
    "Destination IP", "Dst IP",
    "Destination Port", "Dst Port",
    "Timestamp",
]

# Adversarial testing configuration (SAME AS scenario2.md)
N_ADV_SAMPLES = 500

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

# =============================================================================
# MORPHIDS MODEL ARCHITECTURES (Deep Learning)
# =============================================================================

class MLPDropout(nn.Module):
    """Deep MLP with dropout for IDS classification."""
    def __init__(self, d_in, d_out=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(32, d_out)
        )
    def forward(self, x): return self.net(x)


class DeepCNN(nn.Module):
    """1D CNN for IDS classification."""
    def __init__(self, d_in, d_out=2):
        super().__init__()
        self.proj = nn.Linear(d_in, 64)
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, d_out))
    def forward(self, x):
        x = F.relu(self.proj(x)).unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        return self.fc(self.pool(x).squeeze(-1))


class DeepTCN(nn.Module):
    """Temporal Convolutional Network for IDS classification."""
    def __init__(self, d_in, d_out=2):
        super().__init__()
        self.input = nn.Linear(d_in, 64)
        self.tcn1 = nn.Conv1d(1, 16, kernel_size=3, padding=2, dilation=2)
        self.tcn2 = nn.Conv1d(16, 32, kernel_size=3, padding=4, dilation=4)
        self.tcn3 = nn.Conv1d(32, 32, kernel_size=3, padding=8, dilation=8)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, d_out))
    def forward(self, x):
        x = F.relu(self.input(x)).unsqueeze(1)
        x = F.relu(self.tcn1(x))
        x = F.relu(self.tcn2(x))
        x = F.relu(self.tcn3(x))
        return self.fc(self.pool(x).squeeze(-1))


class BiLSTMAttention(nn.Module):
    """Bidirectional LSTM with attention mechanism for IDS classification."""
    def __init__(self, d_in: int, n_classes: int = 2, p: float = 0.2, hidden_size: int = 128, num_layers: int = 2, num_heads: int = 4):
        super().__init__()
        self.hidden_size = hidden_size
        self.proj_size = 64
        self.input_proj = nn.Sequential(nn.Linear(d_in, self.proj_size), nn.LayerNorm(self.proj_size), nn.ReLU(), nn.Dropout(p))
        self.lstm = nn.LSTM(input_size=self.proj_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True, bidirectional=True, dropout=p if num_layers > 1 else 0)
        self.attention = nn.MultiheadAttention(embed_dim=hidden_size * 2, num_heads=num_heads, dropout=p, batch_first=True)
        self.classifier = nn.Sequential(nn.Linear(hidden_size * 2, hidden_size), nn.ReLU(), nn.Dropout(p), nn.Linear(hidden_size, n_classes))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.input_proj(x).unsqueeze(1)
        lstm_out, _ = self.lstm(x)
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        return self.classifier(attn_out.mean(dim=1))


# =============================================================================
# PYTORCH MODEL WRAPPER (for Apollon compatibility)
# =============================================================================

class PyTorchModelWrapper:
    """Wrapper to make PyTorch models compatible with sklearn-like interface."""
    
    def __init__(self, model_class, d_in, d_out=2, device=None, epochs=50, batch_size=1024, lr=0.001):
        self.model_class = model_class
        self.d_in = d_in
        self.d_out = d_out
        self.device = device if device else DEVICE
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.model = None
        self.class_weights = None
        
    def fit(self, X, y):
        """Train the model."""
        X_tensor = torch.FloatTensor(X).to(self.device)
        y_tensor = torch.LongTensor(y).to(self.device)
        class_weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
        self.class_weights = torch.FloatTensor(class_weights).to(self.device)
        self.model = self.model_class(self.d_in, self.d_out).to(self.device)
        dataset = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        criterion = nn.CrossEntropyLoss(weight=self.class_weights)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        self.model.train()
        for epoch in range(self.epochs):
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
        return self
    
    def predict(self, X):
        """Predict class labels with batched processing."""
        self.model.eval()
        all_preds = []
        with torch.no_grad():
            for i in range(0, len(X), self.batch_size):
                batch_X = torch.FloatTensor(X[i:i+self.batch_size]).to(self.device)
                outputs = self.model(batch_X)
                _, predicted = torch.max(outputs, 1)
                all_preds.append(predicted.cpu())
                del batch_X, outputs
                if torch.cuda.is_available(): torch.cuda.empty_cache()
        return torch.cat(all_preds).numpy()
    
    def predict_proba(self, X):
        """Predict class probabilities."""
        self.model.eval()
        all_probs = []
        with torch.no_grad():
            for i in range(0, len(X), self.batch_size):
                batch_X = torch.FloatTensor(X[i:i+self.batch_size]).to(self.device)
                outputs = self.model(batch_X)
                probs = F.softmax(outputs, dim=1)
                all_probs.append(probs.cpu())
                del batch_X, outputs
                if torch.cuda.is_available(): torch.cuda.empty_cache()
        return torch.cat(all_probs).numpy()


# =============================================================================
# APOLLON: MAB + Thompson Sampling with MorphIDS Deep Learning Pool
# =============================================================================

class MultiArmedBanditThompsonSampling:
    """
    APOLLON with MorphIDS Deep Learning Pool.
    
    Uses Multi-Armed Bandits (MAB) with Thompson Sampling to dynamically
    select the optimal Deep Learning classifier for each input.
    Arms: MLP, CNN, TCN, BiLSTM-Attention (MorphIDS pool)
    """
    
    def __init__(self, d_in: int, n_arms: int = 4, n_clusters: int = 2):
        self.d_in = d_in
        self.n_arms = n_arms
        self.n_clusters = n_clusters
        np.random.seed(RANDOM_SEED)
        torch.manual_seed(RANDOM_SEED)
        
        # Initialize MorphIDS Deep Learning arms
        self.arms = [
            PyTorchModelWrapper(MLPDropout, d_in, 2, DEVICE, epochs=30, batch_size=1024),
            PyTorchModelWrapper(DeepCNN, d_in, 2, DEVICE, epochs=30, batch_size=1024),
            PyTorchModelWrapper(DeepTCN, d_in, 2, DEVICE, epochs=30, batch_size=1024),
            PyTorchModelWrapper(BiLSTMAttention, d_in, 2, DEVICE, epochs=30, batch_size=1024),
        ]
        self.arm_names = ['MLP', 'CNN', 'TCN', 'BiLSTM-Attention']
        
        self.cluster_centers = None
        self.cluster_assignments = None
        self.reward_sums = {cluster: np.zeros(n_arms) for cluster in range(n_clusters)}
        self.alpha = np.ones(self.n_arms)
        self.beta = np.ones(self.n_arms)
        self.arm_selections = []
        self.total_predictions = 0
    
    def train(self, X_train: np.ndarray, y_train: np.ndarray) -> None:
        """Train the MAB with KMeans clustering and MorphIDS DL arms."""
        logger.info(f"Training APOLLON MAB with {self.n_clusters} clusters and {self.n_arms} DL arms...")
        
        kmeans = KMeans(n_clusters=self.n_clusters, random_state=RANDOM_SEED, n_init=10)
        self.cluster_assignments = kmeans.fit_predict(X_train)
        self.cluster_centers = kmeans.cluster_centers_
        
        for i in range(self.n_clusters):
            cluster_size = np.sum(self.cluster_assignments == i)
            logger.info(f'Cluster {i}: {cluster_size:,} samples')
        
        for arm_idx, arm in enumerate(self.arms):
            logger.info(f'Training arm {arm_idx} ({self.arm_names[arm_idx]})...')
            arm.fit(X_train, y_train)
        
        for i in range(self.n_clusters):
            cluster_mask = self.cluster_assignments == i
            cluster_X = X_train[cluster_mask]
            cluster_y = y_train[cluster_mask]
            for arm_idx, arm in enumerate(self.arms):
                if len(cluster_X) > 0:
                    predictions = arm.predict(cluster_X)
                    accuracy = np.mean(predictions == cluster_y)
                    self.reward_sums[i][arm_idx] = accuracy
                    logger.info(f'  Arm {arm_idx} on cluster {i}: accuracy={accuracy:.4f}')
        
        logger.info("APOLLON MAB training complete!")
    
    def select_arm(self, cluster: int) -> int:
        """Select arm using Thompson Sampling with Beta distribution."""
        theta = np.zeros(self.n_arms)
        for arm in range(self.n_arms):
            theta[arm] = np.random.beta(
                self.alpha[arm] + self.reward_sums[cluster][arm],
                self.beta[arm] + 1 - self.reward_sums[cluster][arm]
            )
        return int(np.argmax(theta))
    
    def predict(self, X_test: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Predict using MAB-selected arms."""
        arms = np.zeros(len(X_test), dtype=int)
        for i in range(len(X_test)):
            cluster = np.argmin(np.linalg.norm(self.cluster_centers - X_test[i], axis=1))
            arms[i] = self.select_arm(cluster)
        self.arm_selections.extend(arms.tolist())
        
        y_pred = np.zeros(len(X_test))
        for arm in range(self.n_arms):
            arm_mask = arms == arm
            if np.any(arm_mask):
                y_pred[arm_mask] = self.arms[arm].predict(X_test[arm_mask])
        
        self.total_predictions += len(X_test)
        return y_pred, arms
    
    def predict_proba(self, X_test: np.ndarray) -> np.ndarray:
        """Get prediction probabilities for AUC calculation."""
        probs = np.zeros((len(X_test), 2))
        for i in range(len(X_test)):
            cluster = np.argmin(np.linalg.norm(self.cluster_centers - X_test[i:i+1], axis=1))
            arm = self.select_arm(cluster)
            if hasattr(self.arms[arm], 'predict_proba'):
                probs[i] = self.arms[arm].predict_proba(X_test[i:i+1])[0]
            else:
                pred = self.arms[arm].predict(X_test[i:i+1])[0]
                probs[i] = [1 - pred, pred]
        return probs
    
    def get_stats(self) -> Dict:
        """Get MAB statistics."""
        arm_counts = [0] * self.n_arms
        for arm in self.arm_selections:
            arm_counts[arm] += 1
        return {
            'arm_counts': arm_counts,
            'arm_names': self.arm_names,
            'total_predictions': self.total_predictions,
            'reward_sums': {k: v.tolist() for k, v in self.reward_sums.items()},
        }


# =============================================================================
# DL MODEL POOL FOR DIRECT ATTACK (MorphIDS Models)
# =============================================================================

def train_dl_model_pool(X_train: np.ndarray, y_train: np.ndarray, input_dim: int) -> Dict:
    """
    Train individual DL models (MorphIDS pool) for direct adversarial attack.
    """
    logger.info("Training DL model pool for direct attack...")
    
    models = {
        'MLP': PyTorchModelWrapper(MLPDropout, input_dim, 2, DEVICE, epochs=30, batch_size=1024),
        'CNN': PyTorchModelWrapper(DeepCNN, input_dim, 2, DEVICE, epochs=30, batch_size=1024),
        'TCN': PyTorchModelWrapper(DeepTCN, input_dim, 2, DEVICE, epochs=30, batch_size=1024),
        'BiLSTM': PyTorchModelWrapper(BiLSTMAttention, input_dim, 2, DEVICE, epochs=30, batch_size=1024),
    }
    
    for name, model in models.items():
        logger.info(f"Training {name}...")
        model.fit(X_train, y_train)
    
    logger.info("DL model pool training complete!")
    return models


# =============================================================================
# DATA LOADING (Same as scenario2.md)
# =============================================================================

def load_cicids2017_full():
    """
    Load CIC-IDS2017 FULL dataset (benign + attack).
    
    EXACTLY MATCHING scenario2.md preprocessing:
    - Undersampling to 60:40 ratio
    - Same train/val/test split ratios
    - Same drop_duplicates
    - Same scaler loading logic
    """
    logger.info("\n" + "="*80)
    logger.info("LOADING CIC-IDS2017 DATASET (FULL - BENIGN + ATTACKS)")
    logger.info("="*80)
    
    # Load CSV files
    individual_dfs = []
    for idx, csv_file in enumerate(CSV_FILES, 1):
        file_path = os.path.join(DATA_DIR, csv_file)
        logger.info(f"  [{idx}/{len(CSV_FILES)}] Loading: {csv_file}")
        df = pd.read_csv(file_path, sep=',', encoding='utf-8', low_memory=False)
        individual_dfs.append(df)
    
    # Preprocess each dataframe (MATCHING scenario2.md)
    for df in individual_dfs:
        df.columns = df.columns.str.strip()
        for col in DROP_COLS:
            if col in df.columns:
                df.drop(columns=[col], inplace=True, errors='ignore')
        if 'Label' in df.columns:
            df['Label'] = df['Label'].replace({'BENIGN': 'Benign'})
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.dropna(inplace=True)
        df.drop_duplicates(inplace=True)
        df.reset_index(drop=True, inplace=True)
    
    # Concatenate
    df_data = pd.concat(individual_dfs, axis=0, ignore_index=True)
    df_data.dropna(inplace=True)
    df_data.drop_duplicates(inplace=True)
    df_data.reset_index(drop=True, inplace=True)
    
    logger.info(f"Combined dataset shape: {df_data.shape}")
    
    # Label distribution
    logger.info("\nLabel distribution:")
    for label, count in df_data['Label'].value_counts().items():
        logger.info(f"  {label}: {count:,}")
    
    # Extract features and binarize labels
    X = df_data.drop('Label', axis=1).copy()
    y = df_data['Label'].map({'Benign': 0}).fillna(1).astype(int)
    
    # Save feature names
    feature_names = list(X.columns)
    
    logger.info(f"\nOriginal Binary distribution: Benign={sum(y==0):,}, Attack={sum(y==1):,}")
    
    # =========================================================================
    # UNDERSAMPLING: Balance to 60:40 ratio (Benign:Attack)
    # MATCHING scenario2.md exactly
    # =========================================================================
    n_attack = sum(y == 1)
    target_benign = int(n_attack * 1.5)  # 60:40 ratio
    
    benign_idx = np.where(y == 0)[0]
    attack_idx = np.where(y == 1)[0]
    
    # Random sample benign to target size
    np.random.seed(RANDOM_SEED)
    benign_sampled_idx = np.random.choice(benign_idx, size=target_benign, replace=False)
    
    # Combine sampled benign + all attack
    balanced_idx = np.concatenate([benign_sampled_idx, attack_idx])
    np.random.shuffle(balanced_idx)
    
    X = X.iloc[balanced_idx].reset_index(drop=True)
    y = y.iloc[balanced_idx].reset_index(drop=True)
    
    logger.info(f"\n[UNDERSAMPLING] Balanced to 60:40 ratio:")
    logger.info(f"  Benign: {sum(y==0):,} (reduced from {len(benign_idx):,})")
    logger.info(f"  Attack: {sum(y==1):,} (kept all)")
    logger.info(f"  Ratio: {sum(y==0)/len(y)*100:.1f}:{sum(y==1)/len(y)*100:.1f}")
    
    # Stratified split (MATCHING scenario2.md: 75% train, 10% val, 15% test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, stratify=y, test_size=0.25, random_state=RANDOM_SEED, shuffle=True
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, stratify=y_temp, test_size=0.60, random_state=RANDOM_SEED, shuffle=True
    )
    
    logger.info(f"\nDataset split (after balancing):")
    logger.info(f"  Train: {len(X_train):,} (Benign: {sum(y_train==0):,}, Attack: {sum(y_train==1):,})")
    logger.info(f"  Val:   {len(X_val):,} (Benign: {sum(y_val==0):,}, Attack: {sum(y_val==1):,})")
    logger.info(f"  Test:  {len(X_test):,} (Benign: {sum(y_test==0):,}, Attack: {sum(y_test==1):,})")
    
    # =========================================================================
    # StandardScaler (MATCHING scenario2.md - load from file if exists)
    # =========================================================================
    import joblib
    scaler_path = os.path.join(MODEL_DIR, 'scaler_standard.pkl')
    if os.path.exists(scaler_path):
        logger.info(f"Loading StandardScaler from {scaler_path}")
        scaler = joblib.load(scaler_path)
        X_train_scaled = scaler.transform(X_train).astype(np.float32)
    else:
        logger.info("Fitting new StandardScaler (Z-score normalization)...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
        os.makedirs(MODEL_DIR, exist_ok=True)
        joblib.dump(scaler, scaler_path)
    
    X_val_scaled = scaler.transform(X_val).astype(np.float32)
    X_test_scaled = scaler.transform(X_test).astype(np.float32)
    
    # Compute actual min/max for ART clip_values
    clip_min = float(X_train_scaled.min())
    clip_max = float(X_train_scaled.max())
    logger.info(f"StandardScaler range: [{clip_min:.4f}, {clip_max:.4f}]")
    
    input_dim = X_train_scaled.shape[1]
    
    return (X_train_scaled, X_val_scaled, X_test_scaled,
            y_train.values, y_val.values, y_test.values,
            scaler, input_dim, feature_names,
            (clip_min, clip_max))


# =============================================================================
# ART ADVERSARIAL GENERATION - DIRECT ATTACK ON PYTORCH DL MODELS
# =============================================================================

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import ZooAttack, HopSkipJump


class ARTModelWrapper(nn.Module):
    """Wrapper to make PyTorchModelWrapper compatible with ART.
    
    Handles dtype conversion: ART passes Double (float64), PyTorch expects Float (float32).
    """
    def __init__(self, pytorch_wrapper):
        super().__init__()
        self.inner_model = pytorch_wrapper.model
    
    def forward(self, x):
        # Convert to float32 if needed (ART sometimes passes float64)
        if x.dtype == torch.float64:
            x = x.float()
        return self.inner_model(x)


def create_art_classifier(pytorch_wrapper, clip_values, input_dim):
    """Create ART PyTorchClassifier from PyTorchModelWrapper."""
    # Wrap model with dtype conversion handler
    wrapped_model = ARTModelWrapper(pytorch_wrapper)
    wrapped_model.eval()
    
    # Create ART classifier with wrapped model
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(wrapped_model.parameters(), lr=0.001)
    
    art_classifier = PyTorchClassifier(
        model=wrapped_model,
        loss=criterion,
        optimizer=optimizer,
        input_shape=(input_dim,),
        nb_classes=2,
        clip_values=clip_values,
        device_type="gpu" if torch.cuda.is_available() else "cpu"
    )
    return art_classifier


def generate_adversarial_samples_zoo_pytorch(pytorch_wrapper, X_clean, y_clean, clip_values=(0, 1), 
                                             n_samples=N_ADV_SAMPLES, seed=RANDOM_SEED, input_dim=76):
    """
    Generate adversarial samples using ZOO attack on PyTorch DL model.
    """
    np.random.seed(seed)
    
    if len(X_clean) > n_samples:
        idx = np.random.choice(len(X_clean), n_samples, replace=False)
        X_subset = X_clean[idx].copy()
        y_subset = y_clean[idx].copy()
    else:
        X_subset = X_clean.copy()
        y_subset = y_clean.copy()
    
    # Create ART classifier for PyTorch model
    classifier = create_art_classifier(pytorch_wrapper, clip_values, input_dim)
    
    # ZOO attack
    attack = ZooAttack(
        classifier=classifier,
        targeted=True,
        nb_parallel=60,
        confidence=0.0,
        learning_rate=0.01,
        max_iter=100,
        binary_search_steps=1,
        initial_const=1e-3,
        abort_early=True,
        use_resize=False,
        use_importance=False,
    )
    
    start_time = time.time()
    
    # Target label = 0 (Benign) - one-hot encoding for PyTorch
    y_target = np.zeros((len(X_subset), 2))
    y_target[:, 0] = 1  # One-hot for class 0
    X_adv = attack.generate(X_subset.astype(np.float32), y_target)
    
    elapsed = time.time() - start_time
    
    # Calculate success rate
    pred_adv = pytorch_wrapper.predict(X_adv)
    attack_success = (pred_adv == 0).sum()
    attack_success_rate = attack_success / len(X_subset) * 100
    
    logger.info(f"Success rate of ZOO attack: {attack_success_rate:.2f}%")
    
    return X_adv, y_subset, {
        'attack_type': 'ZOO',
        'n_samples': len(X_subset),
        'success_rate': attack_success_rate,
        'elapsed_time': elapsed
    }


def generate_adversarial_samples_hopskipjump_pytorch(pytorch_wrapper, X_clean, y_clean, clip_values=(0, 1),
                                                     n_samples=100, seed=RANDOM_SEED, input_dim=76):
    """Generate adversarial samples using HopSkipJump attack on PyTorch DL model."""
    np.random.seed(seed)
    
    if len(X_clean) > n_samples:
        idx = np.random.choice(len(X_clean), n_samples, replace=False)
        X_subset = X_clean[idx].copy()
        y_subset = y_clean[idx].copy()
    else:
        X_subset = X_clean.copy()
        y_subset = y_clean.copy()
    
    classifier = create_art_classifier(pytorch_wrapper, clip_values, input_dim)
    
    attack = HopSkipJump(classifier=classifier, targeted=True, max_iter=50, max_eval=1000)
    
    start_time = time.time()
    
    # Target label = 0 (Benign) - one-hot encoding
    y_target = np.zeros((len(X_subset), 2))
    y_target[:, 0] = 1
    X_adv = attack.generate(X_subset.astype(np.float32), y_target)
    
    elapsed = time.time() - start_time
    
    pred_adv = pytorch_wrapper.predict(X_adv)
    attack_success = (pred_adv == 0).sum()
    attack_success_rate = attack_success / len(X_subset) * 100
    
    logger.info(f"Success rate of HSJA attack: {attack_success_rate:.2f}%")
    
    return X_adv, y_subset, {
        'attack_type': 'HopSkipJump',
        'n_samples': len(X_subset),
        'success_rate': attack_success_rate,
        'elapsed_time': elapsed
    }


# =============================================================================
# EVALUATION PIPELINE
# =============================================================================

def evaluate_model(model, X_test, y_test, model_name="Model"):
    """Evaluate a single model."""
    predictions = model.predict(X_test)
    
    try:
        if hasattr(model, 'predict_proba'):
            probs = model.predict_proba(X_test)[:, 1]
        else:
            probs = predictions
        auc = roc_auc_score(y_test, probs)
    except:
        auc = 0.5
    
    return {
        'accuracy': accuracy_score(y_test, predictions),
        'detection_rate': recall_score(y_test, predictions, zero_division=0),
        'f1': f1_score(y_test, predictions, zero_division=0),
        'precision': precision_score(y_test, predictions, zero_division=0),
        'auc': auc,
        'cm': confusion_matrix(y_test, predictions, labels=[0, 1]),
    }


def evaluate_apollon_adversarial_robustness(
    model_pool: Dict,
    apollon: MultiArmedBanditThompsonSampling,
    X_attack: np.ndarray,
    y_attack: np.ndarray,
    X_benign: np.ndarray,
    y_benign: np.ndarray,
    clip_values: Tuple[float, float] = (0, 1),
    input_dim: int = 76,
    n_zoo_attack: int = 500,
    n_zoo_benign: int = 500,
    n_hsja_attack: int = 100,
    n_hsja_benign: int = 100,
    seed: int = RANDOM_SEED
) -> Dict:
    """
    Evaluate APOLLON (MAB) against ZOO and HopSkipJump attacks.
    
    DIRECT ATTACK: Attacks target the same DL models used in Apollon.
    """
    print("\n" + "="*80)
    print("APOLLON (MAB + Thompson Sampling) ADVERSARIAL EVALUATION")
    print("="*80)
    print(f"DIRECT ATTACK on DL Models: {list(model_pool.keys())}")
    print(f"ZOO Attack:  {n_zoo_attack} attack + {n_zoo_benign} benign = {n_zoo_attack + n_zoo_benign} per model")
    print(f"HSJA Attack: {n_hsja_attack} attack + {n_hsja_benign} benign = {n_hsja_attack + n_hsja_benign} per model")
    
    np.random.seed(seed)
    
    model_names = list(model_pool.keys())
    
    # =========================================================================
    # PHASE 1: ZOO ATTACK EVALUATION
    # =========================================================================
    print(f"\n----- ZOO Evaluation -----")
    
    zoo_results = {}
    zoo_apollon_accs, zoo_apollon_drs, zoo_apollon_f1s = [], [], []
    
    # Sample benign ONCE before loop (matching scenario2.md)
    np.random.seed(seed)
    zoo_benign_idx = np.random.choice(len(X_benign), min(n_zoo_benign, len(X_benign)), replace=False)
    X_zoo_benign = X_benign[zoo_benign_idx]
    y_zoo_benign = y_benign[zoo_benign_idx]
    
    for model_idx, model_name in enumerate(model_names):
        model = model_pool[model_name]
        
        # Generate ZOO adversarial samples targeting THIS DL model directly
        X_adv, y_adv, zoo_stats = generate_adversarial_samples_zoo_pytorch(
            pytorch_wrapper=model,
            X_clean=X_attack,
            y_clean=y_attack,
            clip_values=clip_values,
            n_samples=n_zoo_attack,
            seed=seed + model_idx,
            input_dim=input_dim
        )
        
        # Create test set: adversarial + benign
        X_test = np.vstack([X_adv, X_zoo_benign])
        y_test = np.concatenate([y_adv, y_zoo_benign])
        
        # Evaluate single model (target)
        model_metrics = evaluate_model(model, X_test, y_test, model_name)
        
        # Evaluate APOLLON
        apollon_preds, apollon_arms = apollon.predict(X_test)
        try:
            apollon_probs = apollon.predict_proba(X_test)[:, 1]
            apollon_auc = roc_auc_score(y_test, apollon_probs)
        except:
            apollon_auc = 0.5
        
        apollon_metrics = {
            'accuracy': accuracy_score(y_test, apollon_preds),
            'detection_rate': recall_score(y_test, apollon_preds, zero_division=0),
            'f1': f1_score(y_test, apollon_preds, zero_division=0),
            'precision': precision_score(y_test, apollon_preds, zero_division=0),
            'auc': apollon_auc,
            'cm': confusion_matrix(y_test, apollon_preds, labels=[0, 1]),
        }
        
        zoo_results[model_name] = {
            'attack_success_rate': zoo_stats['success_rate'],
            'model': model_metrics,
            'apollon': apollon_metrics,
        }
        
        zoo_apollon_accs.append(apollon_metrics['accuracy'])
        zoo_apollon_drs.append(apollon_metrics['detection_rate'])
        zoo_apollon_f1s.append(apollon_metrics['f1'])
        
        print(f"====================> Model: {model_name.upper()}")
        print("---------- Single Model (Target)")
        print(f"Accuracy:  {model_metrics['accuracy']:.5f}")
        print(f"Detection Rate:  {model_metrics['detection_rate']:.5f}")
        print(f"F1 Score:  {model_metrics['f1']:.5f}")
        print(f"ROC AUC Score:  {model_metrics['auc']:.5f}")
        print("---------- APOLLON (MAB + Thompson Sampling)")
        tn, fp, fn, tp = apollon_metrics['cm'].ravel()
        print(f"Confusion Matrix: [TN={tn} FP={fp}] [FN={fn} TP={tp}]")
        print(f"Accuracy:  {apollon_metrics['accuracy']:.5f}")
        print(f"Detection Rate:  {apollon_metrics['detection_rate']:.5f}")
        print(f"F1 Score:  {apollon_metrics['f1']:.5f}")
        print(f"ROC AUC Score:  {apollon_metrics['auc']:.5f}")
    
    # =========================================================================
    # PHASE 2: HSJA ATTACK EVALUATION
    # =========================================================================
    print(f"\n----- HSJA Evaluation -----")
    
    hsja_results = {}
    hsja_apollon_accs, hsja_apollon_drs, hsja_apollon_f1s = [], [], []
    
    # Sample benign ONCE
    np.random.seed(seed + 2000)
    hsja_benign_idx = np.random.choice(len(X_benign), min(n_hsja_benign, len(X_benign)), replace=False)
    X_hsja_benign = X_benign[hsja_benign_idx]
    y_hsja_benign = y_benign[hsja_benign_idx]
    
    for model_idx, model_name in enumerate(model_names):
        model = model_pool[model_name]
        
        # Generate HSJA adversarial samples targeting THIS DL model directly
        X_adv, y_adv, hsja_stats = generate_adversarial_samples_hopskipjump_pytorch(
            pytorch_wrapper=model,
            X_clean=X_attack,
            y_clean=y_attack,
            clip_values=clip_values,
            n_samples=n_hsja_attack,
            seed=seed + 1000 + model_idx,
            input_dim=input_dim
        )
        
        X_test = np.vstack([X_adv, X_hsja_benign])
        y_test = np.concatenate([y_adv, y_hsja_benign])
        
        model_metrics = evaluate_model(model, X_test, y_test, model_name)
        
        apollon_preds, apollon_arms = apollon.predict(X_test)
        try:
            apollon_probs = apollon.predict_proba(X_test)[:, 1]
            apollon_auc = roc_auc_score(y_test, apollon_probs)
        except:
            apollon_auc = 0.5
        
        apollon_metrics = {
            'accuracy': accuracy_score(y_test, apollon_preds),
            'detection_rate': recall_score(y_test, apollon_preds, zero_division=0),
            'f1': f1_score(y_test, apollon_preds, zero_division=0),
            'precision': precision_score(y_test, apollon_preds, zero_division=0),
            'auc': apollon_auc,
            'cm': confusion_matrix(y_test, apollon_preds, labels=[0, 1]),
        }
        
        hsja_results[model_name] = {
            'attack_success_rate': hsja_stats['success_rate'],
            'model': model_metrics,
            'apollon': apollon_metrics,
        }
        
        hsja_apollon_accs.append(apollon_metrics['accuracy'])
        hsja_apollon_drs.append(apollon_metrics['detection_rate'])
        hsja_apollon_f1s.append(apollon_metrics['f1'])
        
        print(f"====================> Model: {model_name.upper()}")
        print("---------- Single Model (Target)")
        print(f"Accuracy:  {model_metrics['accuracy']:.5f}")
        print(f"Detection Rate:  {model_metrics['detection_rate']:.5f}")
        print(f"F1 Score:  {model_metrics['f1']:.5f}")
        print(f"ROC AUC Score:  {model_metrics['auc']:.5f}")
        print("---------- APOLLON (MAB + Thompson Sampling)")
        tn, fp, fn, tp = apollon_metrics['cm'].ravel()
        print(f"Confusion Matrix: [TN={tn} FP={fp}] [FN={fn} TP={tp}]")
        print(f"Accuracy:  {apollon_metrics['accuracy']:.5f}")
        print(f"Detection Rate:  {apollon_metrics['detection_rate']:.5f}")
        print(f"F1 Score:  {apollon_metrics['f1']:.5f}")
        print(f"ROC AUC Score:  {apollon_metrics['auc']:.5f}")
    
    # =========================================================================
    # SUMMARY
    # =========================================================================
    print("\n" + "="*80)
    print("APOLLON FINAL RESULTS SUMMARY")
    print("="*80)
    
    print("\n[ZOO Attack Results] (500 attack + 500 benign per model)")
    print("-" * 100)
    print(f"{'Target':<10} {'ZOO Success':>12} {'Model Acc':>12} {'Model DR':>10} {'APOLLON Acc':>12} {'APOLLON DR':>12} {'APOLLON F1':>12}")
    print("-" * 100)
    for name in model_names:
        r = zoo_results[name]
        print(f"{name.upper():<10} {r['attack_success_rate']:>11.1f}% {r['model']['accuracy']*100:>11.1f}% {r['model']['detection_rate']*100:>9.1f}% {r['apollon']['accuracy']*100:>11.1f}% {r['apollon']['detection_rate']*100:>11.1f}% {r['apollon']['f1']*100:>11.1f}%")
    
    print("\n[HSJA Attack Results] (100 attack + 100 benign per model)")
    print("-" * 100)
    print(f"{'Target':<10} {'HSJA Success':>12} {'Model Acc':>12} {'Model DR':>10} {'APOLLON Acc':>12} {'APOLLON DR':>12} {'APOLLON F1':>12}")
    print("-" * 100)
    for name in model_names:
        r = hsja_results[name]
        print(f"{name.upper():<10} {r['attack_success_rate']:>11.1f}% {r['model']['accuracy']*100:>11.1f}% {r['model']['detection_rate']*100:>9.1f}% {r['apollon']['accuracy']*100:>11.1f}% {r['apollon']['detection_rate']*100:>11.1f}% {r['apollon']['f1']*100:>11.1f}%")
    
    print("\n" + "="*80)
    print("APOLLON AVERAGE PERFORMANCE")
    print("="*80)
    print(f"{'Attack':<12} {'Accuracy':>12} {'Detection Rate':>16} {'F1 Score':>12}")
    print("-" * 56)
    print(f"{'ZOO':<12} {np.mean(zoo_apollon_accs)*100:>11.2f}% {np.mean(zoo_apollon_drs)*100:>15.2f}% {np.mean(zoo_apollon_f1s)*100:>11.2f}%")
    print(f"{'HSJA':<12} {np.mean(hsja_apollon_accs)*100:>11.2f}% {np.mean(hsja_apollon_drs)*100:>15.2f}% {np.mean(hsja_apollon_f1s)*100:>11.2f}%")
    print("="*80)
    
    # Print MAB statistics
    mab_stats = apollon.get_stats()
    print("\n[APOLLON MAB Statistics]")
    for i, (name, count) in enumerate(zip(mab_stats['arm_names'], mab_stats['arm_counts'])):
        print(f"  {name}: {count} selections ({count/mab_stats['total_predictions']*100:.1f}%)")
    
    return {
        'zoo': {
            'per_model': zoo_results,
            'average': {
                'accuracy': np.mean(zoo_apollon_accs),
                'detection_rate': np.mean(zoo_apollon_drs),
                'f1': np.mean(zoo_apollon_f1s),
            }
        },
        'hsja': {
            'per_model': hsja_results,
            'average': {
                'accuracy': np.mean(hsja_apollon_accs),
                'detection_rate': np.mean(hsja_apollon_drs),
                'f1': np.mean(hsja_apollon_f1s),
            }
        },
        'model_names': model_names,
        'mab_stats': mab_stats,
    }


# =============================================================================
# MAIN PIPELINE
# =============================================================================

def run_apollon_evaluation():
    """
    Run APOLLON baseline evaluation (exact replication of zoo.ipynb).
    """
    print("\n" + "="*80)
    print("APOLLON BASELINE EVALUATION PIPELINE")
    print("="*80)
    print("Replicating zoo.ipynb with attack settings from scenario2.md")
    print(f"Models: MLP, CNN, TCN, BiLSTM-Attention (MorphIDS Deep Learning Pool)")
    print(f"MAB: Thompson Sampling with 2 clusters")
    print(f"Attacks: ZOO + HopSkipJump")
    print(f"Random Seed: {RANDOM_SEED}")
    print("="*80)
    
    # Load data
    logger.info("\n[1/4] Loading CIC-IDS2017...")
    (X_train, X_val, X_test,
     y_train, y_val, y_test,
     scaler, input_dim, feature_names,
     clip_values) = load_cicids2017_full()
    
    logger.info(f"Train: {len(X_train):,}, Val: {len(X_val):,}, Test: {len(X_test):,}")
    logger.info(f"Input dim: {input_dim}")
    logger.info(f"Clip values: {clip_values}")
    
    # Train DL model pool for direct attack (same models as Apollon arms)
    logger.info("\n[2/4] Training DL Model Pool for Direct Attack...")
    model_pool = train_dl_model_pool(X_train, y_train, input_dim)
    
    # Train APOLLON MAB with MorphIDS DL Pool
    logger.info("\n[3/4] Training APOLLON (MAB with MorphIDS Deep Learning Pool)...")
    apollon = MultiArmedBanditThompsonSampling(d_in=input_dim, n_arms=4, n_clusters=2)
    apollon.train(X_train, y_train)
    
    # Separate test data
    benign_idx = np.where(y_test == 0)[0]
    attack_idx = np.where(y_test == 1)[0]
    X_benign_test = X_test[benign_idx]
    y_benign_test = y_test[benign_idx]
    X_attack_test = X_test[attack_idx]
    y_attack_test = y_test[attack_idx]
    
    logger.info(f"Test set: {len(X_benign_test):,} benign, {len(X_attack_test):,} attack")
    
    # Run APOLLON evaluation
    logger.info("\n[4/4] Running APOLLON Adversarial Evaluation...")
    
    results = evaluate_apollon_adversarial_robustness(
        model_pool=model_pool,
        apollon=apollon,
        X_attack=X_attack_test,
        y_attack=y_attack_test,
        X_benign=X_benign_test,
        y_benign=y_benign_test,
        clip_values=clip_values,
        input_dim=input_dim,
        n_zoo_attack=500,
        n_zoo_benign=500,
        n_hsja_attack=100,
        n_hsja_benign=100,
        seed=RANDOM_SEED
    )
    
    print("\n" + "="*80)
    print("[OK] APOLLON Baseline Evaluation Complete!")
    print(f"Results saved to: {OUTPUT_DIR}")
    print("="*80)
    
    return results


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    run_apollon_evaluation()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.0 MB/s eta 0:00:00


2026-02-02 12:06:33,341 | INFO | set ART_DATA_PATH to /root/.art/data


🎯 Using Device: cpu


2026-02-02 12:06:35,718 | INFO | 
[1/4] Loading CIC-IDS2017...
2026-02-02 12:06:35,719 | INFO | 
2026-02-02 12:06:35,720 | INFO | LOADING CIC-IDS2017 DATASET (FULL - BENIGN + ATTACKS)
2026-02-02 12:06:35,721 | INFO | ================================================================================
2026-02-02 12:06:35,722 | INFO |   [1/3] Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv



APOLLON BASELINE EVALUATION PIPELINE
Replicating zoo.ipynb with attack settings from scenario2.md
Models: MLP, CNN, TCN, BiLSTM-Attention (MorphIDS Deep Learning Pool)
MAB: Thompson Sampling with 2 clusters
Attacks: ZOO + HopSkipJump
Random Seed: 42


...

2026-02-02 12:58:25,574 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 12:58:27,350 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 12:58:29,064 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 12:58:30,747 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 12:58:32,461 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 12:58:34,157 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 12:58:35,842 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 12:58:37,651 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 12:58:39,507 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 12:58:41,340 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 12:58:43,127 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 12:58:44,939 | INFO | Found initial adversa

====================> Model: TCN
---------- Single Model (Target)
Accuracy:  0.48500
Detection Rate:  0.00000
F1 Score:  0.00000
ROC AUC Score:  0.95400
---------- APOLLON (MAB + Thompson Sampling)
Confusion Matrix: [TN=98 FP=2] [FN=47 TP=53]
Accuracy:  0.75500
Detection Rate:  0.53000
F1 Score:  0.68387
ROC AUC Score:  0.87950


HopSkipJump:   0%|          | 0/100 [00:00<?, ?it/s]

2026-02-02 13:01:20,968 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 13:01:24,367 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 13:01:27,597 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 13:01:30,730 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 13:01:33,924 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 13:01:37,075 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 13:01:40,336 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 13:01:43,613 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 13:01:46,787 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 13:01:50,067 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 13:01:53,174 | INFO | Found initial adversarial image for targeted attack.
2026-02-02 13:01:56,487 | INFO | Found initial adversa

====================> Model: BILSTM
---------- Single Model (Target)
Accuracy:  0.49000
Detection Rate:  0.00000
F1 Score:  0.00000
ROC AUC Score:  0.94040
---------- APOLLON (MAB + Thompson Sampling)
Confusion Matrix: [TN=98 FP=2] [FN=17 TP=83]
Accuracy:  0.90500
Detection Rate:  0.83000
F1 Score:  0.89730
ROC AUC Score:  0.98400

APOLLON FINAL RESULTS SUMMARY

[ZOO Attack Results] (500 attack + 500 benign per model)
----------------------------------------------------------------------------------------------------
Target      ZOO Success    Model Acc   Model DR  APOLLON Acc   APOLLON DR   APOLLON F1
----------------------------------------------------------------------------------------------------
MLP               34.6%        82.2%      65.4%        93.5%        88.4%        93.2%
CNN               33.2%        82.6%      66.8%        94.2%        89.2%        93.9%
TCN               31.4%        83.5%      68.6%        94.8%        91.0%        94.6%
BILSTM            75.6%     